# Data Preprocessing: UNSW-NB15

This notebook performs the data preprocessing stage for the UNSW-NB15 cybersecurity dataset used in the autonomous zero-day detection project.

The goal here is to build a clean preprocessing pipeline without training a model yet. We keep `label` as the binary target for regular attack detection, and `attack_cat` is preserved separately for future zero-day research experiments.

Important: the original CSV files are never modified. We fit the encoder and scaler only on the training data to prevent data leakage.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

project_root = Path.cwd().resolve()
for candidate in [project_root, project_root.parent]:
    if (candidate / 'dataset' / 'UNSW-NB15').exists():
        project_root = candidate
        break

if not (project_root / 'dataset' / 'UNSW-NB15').exists():
    raise FileNotFoundError('The UNSW-NB15 dataset folder was not found. Ensure the project root is correct.')

project_root = project_root.resolve()
data_dir = project_root / 'dataset' / 'UNSW-NB15'
model_dir = project_root / 'models'
model_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / 'UNSW_NB15_training-set.csv'
test_path = data_dir / 'UNSW_NB15_testing-set.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f'Project root: {project_root}')
print(f'Training set path: {train_path}')
print(f'Testing set path: {test_path}')
print(f'\nTrain shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
print(f'Missing values in training set: {int(train_df.isna().sum().sum())}')
print(f'Missing values in testing set: {int(test_df.isna().sum().sum())}')
print(f'\nTraining label distribution:')
print(train_df['label'].value_counts().sort_index().to_string())
print(f'Testing label distribution:')
print(test_df['label'].value_counts().sort_index().to_string())
print(f'\nTraining attack_cat distribution:')
print(train_df['attack_cat'].value_counts().to_string())
print(f'Testing attack_cat distribution:')
print(test_df['attack_cat'].value_counts().to_string())

Project root: C:\Users\Aliya\OneDrive\Desktop\sem 4\Autonomous_Zero_Day_Detection\Autonomous_Zero_Day_Detection
Training set path: C:\Users\Aliya\OneDrive\Desktop\sem 4\Autonomous_Zero_Day_Detection\Autonomous_Zero_Day_Detection\dataset\UNSW-NB15\UNSW_NB15_training-set.csv
Testing set path: C:\Users\Aliya\OneDrive\Desktop\sem 4\Autonomous_Zero_Day_Detection\Autonomous_Zero_Day_Detection\dataset\UNSW-NB15\UNSW_NB15_testing-set.csv

Train shape: (175341, 45)
Test shape: (82332, 45)
Missing values in training set: 0
Missing values in testing set: 0

Training label distribution:
label
0     56000
1    119341
Testing label distribution:
label
0    37000
1    45332

Training attack_cat distribution:
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Testing attack_cat distribution:
attack_cat
No

In [2]:
target_col = 'label'
attack_col = 'attack_cat'

# Keep the binary label as the main supervised target.
# Preserve the attack category separately for later zero-day and federated experiments.
exclude_cols = [target_col, attack_col]
feature_cols = [col for col in train_df.columns if col not in exclude_cols]

print(f'Binary target column: {target_col}')
print(f'Attack category column retained separately: {attack_col}')
print(f'\nFeature columns ({len(feature_cols)}):')
print(feature_cols)
print(f'\nFeature consistency check:')
print(f'Training feature columns match testing feature columns: {list(train_df[feature_cols].columns) == list(test_df[feature_cols].columns)}')
print(f'Training rows: {train_df.shape[0]}')
print(f'Testing rows: {test_df.shape[0]}')
print(f'Numeric features in training set: {train_df[feature_cols].select_dtypes(include=[np.number]).shape[1]}')
print(f'Categorical features in training set: {train_df[feature_cols].select_dtypes(exclude=[np.number]).shape[1]}')

# Keep the original data untouched. We only build training/test feature matrices.
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()
y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

attack_train = train_df[attack_col].copy()
attack_test = test_df[attack_col].copy()

print(f'\nX_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

Binary target column: label
Attack category column retained separately: attack_cat

Feature columns (43):
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']

Feature consistency check:
Training feature columns match testing feature columns: True
Training rows: 175341
Testing rows: 82332
Numeric features in training set: 40
Categorical features in training set: 3

X_train shape: (175341, 43)
X_test shape: (82332, 43)
y_train shape: (175341,)
y_test shape: (82332,)


In [3]:
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

print('Categorical features to encode:')
print(categorical_features)
print('\nNumerical features to scale:')
print(numerical_features)

# A simple preprocessing pipeline: one-hot encode categorical features and standardize numeric features.
# The encoder and scaler are fit only on the training data.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ]
)

X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)

print(f'\nProcessed training matrix shape: {X_train_pre.shape}')
print(f'Processed testing matrix shape: {X_test_pre.shape}')
print('Processed feature matrix contains no missing values:', not np.isnan(X_train_pre).any(), not np.isnan(X_test_pre).any())

# Save the fitted preprocessing pipeline for later use.
pickle_path = model_dir / 'preprocessing_pipeline.pkl'
with open(pickle_path, 'wb') as f:
    pickle.dump(preprocessor, f)
print(f'Preprocessing pipeline saved to: {pickle_path}')

Categorical features to encode:
['proto', 'service', 'state']

Numerical features to scale:
['id', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']



Processed training matrix shape: (175341, 195)
Processed testing matrix shape: (82332, 195)
Processed feature matrix contains no missing values: True True
Preprocessing pipeline saved to: C:\Users\Aliya\OneDrive\Desktop\sem 4\Autonomous_Zero_Day_Detection\Autonomous_Zero_Day_Detection\models\preprocessing_pipeline.pkl


In [4]:
train_class_counts = y_train.value_counts().sort_index()
test_class_counts = y_test.value_counts().sort_index()

train_class_pct = y_train.value_counts(normalize=True).sort_index() * 100
test_class_pct = y_test.value_counts(normalize=True).sort_index() * 100

print('Training binary target distribution:')
print(pd.concat([train_class_counts, train_class_pct], axis=1).rename(columns={0: 'Count', 1: 'Percentage (%)'}))
print('\nTesting binary target distribution:')
print(pd.concat([test_class_counts, test_class_pct], axis=1).rename(columns={0: 'Count', 1: 'Percentage (%)'}))

print('\nTraining attack category distribution after preprocessing (retained separately):')
print(attack_train.value_counts().to_string())
print('\nTesting attack category distribution after preprocessing (retained separately):')
print(attack_test.value_counts().to_string())

Training binary target distribution:
        count  proportion
label                    
0       56000   31.937767
1      119341   68.062233

Testing binary target distribution:
       count  proportion
label                   
0      37000   44.939999
1      45332   55.060001

Training attack category distribution after preprocessing (retained separately):
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130

Testing attack category distribution after preprocessing (retained separately):
attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
